In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Block 1 — Imports and paths
# Run this first after downloading artifacts_v2 from Drive
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pickle
import pandas as pd
from datetime import datetime
from pathlib import Path

# ── Set your paths here ───────────────────────────────────────────────────────
ARTIFACTS_DIR   = './artifacts_v2'          # folder with embeddings.npy + metadata.pkl + jd_embedding.npy
OUTPUT_DIR      = './outputs_v2'
CANDIDATES_PATH = './candidates.jsonl'      # only needed for full summary enrichment check

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Load artifacts ─────────────────────────────────────────────────────────────
print('Loading artifacts...')
embeddings = np.load(f'{ARTIFACTS_DIR}/embeddings.npy')

with open(f'{ARTIFACTS_DIR}/metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

# R1: Load real JD embedding (not centroid proxy)
jd_embedding = np.load(f'{ARTIFACTS_DIR}/jd_embedding.npy').astype(np.float32)
jd_embedding /= max(np.linalg.norm(jd_embedding), 1e-8)

print(f'✓ embeddings.npy    — shape: {embeddings.shape}')
print(f'✓ metadata.pkl      — {len(metadata):,} candidates')
print(f'✓ jd_embedding.npy  — shape: {jd_embedding.shape} norm: {np.linalg.norm(jd_embedding):.4f}')
print(f'✓ Embedding RAM     : {embeddings.nbytes/1e6:.1f} MB')

# Verify v2 fields present in metadata
m = metadata[0]
print(f'\nv2 metadata fields:')
print(f'  full_summary len  : {len(m.get("full_summary", ""))} chars')
print(f'  avg_tenure_months : {m.get("avg_tenure_months", "MISSING")}')
print(f'  summary_quality   : {m.get("summary_quality", "MISSING")}')
print(f'  company_size_score: {m.get("company_size_score", "MISSING")}')
print(f'  is_faang_only     : {m.get("is_faang_only", "MISSING")}')
print(f'  location_bucket   : {m.get("location_bucket", "MISSING")}')

Loading artifacts...
✓ embeddings.npy    — shape: (100000, 1024)
✓ metadata.pkl      — 100,000 candidates
✓ jd_embedding.npy  — shape: (1024,) norm: 1.0000
✓ Embedding RAM     : 409.6 MB

v2 metadata fields:
  full_summary len  : 634 chars
  avg_tenure_months : 41.0
  summary_quality   : 0.7333333333333333
  company_size_score: 0.5
  is_faang_only     : False
  location_bucket   : outside_india


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Block 2 — Hard filters
# ─────────────────────────────────────────────────────────────────────────────
from datetime import datetime

TODAY = datetime(2026, 6, 1)

def passes_hard_filters(m):
    if not m['verified_email'] and not m['verified_phone']:
        return False, 'both email and phone unverified'
    if not m['open_to_work_flag']:
        return False, 'not open to work'
    if m['english_proficiency'] not in ('professional', 'native', 'fluent', 'full professional'):
        return False, f'english proficiency insufficient'
    if m['notice_period_days'] > 60:
        return False, f'notice period too long ({m["notice_period_days"]}d)'
    if m['location_bucket'] == 'india_no_relocate':
        return False, 'not in preferred city and not willing to relocate'
    if m['industry_flag'] == 'reject':
        return False, f'industry/title mismatch'
    if m['title_bucket'] == 'non_tech':
        return False, f'non-tech title ({m["current_title"]})'
    if m['consulting_pct'] >= 1.0:
        return False, 'entire career at consulting firms'
    if m['is_pure_research']:
        return False, 'pure research, no production evidence'
    if m['is_cv_speech_no_nlp']:
        return False, 'CV/speech only, no NLP/IR'
    if m['is_closed_source']:
        return False, 'closed-source 5+ years, no external validation'
    if m['last_coding_months_ago'] > 18:
        return False, 'no coding in last 18 months'
    if m['has_zero_skill_duration']:
        return False, 'skill with zero duration'
    if m['exp_sum_mismatch_months'] > 6:
        return False, f'experience sum mismatch ({m["exp_sum_mismatch_months"]:.0f} months)'
    return True, ''

print('Applying hard filters...')
surviving      = []
rejected_count = 0
reject_reasons = {}

for i, m in enumerate(metadata):
    passed, reason = passes_hard_filters(m)
    if passed:
        surviving.append(i)
    else:
        rejected_count += 1
        key = reason.split('(')[0].strip()
        reject_reasons[key] = reject_reasons.get(key, 0) + 1

print(f'✓ Passed  : {len(surviving):,}')
print(f'✗ Rejected: {rejected_count:,}')
print(f'\nTop rejection reasons:')
for reason, count in sorted(reject_reasons.items(), key=lambda x: -x[1])[:10]:
    print(f'  {reason}: {count:,}')

# Normalize embeddings
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1, norms)
embeddings_norm = (embeddings / norms).astype(np.float32)
print(f'\n✓ Embeddings normalized')

Applying hard filters...
✓ Passed  : 1,126
✗ Rejected: 98,874

Top rejection reasons:
  not open to work: 57,596
  notice period too long: 19,748
  both email and phone unverified: 10,738
  not in preferred city and not willing to relocate: 3,197
  industry/title mismatch: 3,026
  non-tech title: 2,804
  closed-source 5+ years, no external validation: 700
  entire career at consulting firms: 477
  CV/speech only, no NLP/IR: 336
  no coding in last 18 months: 183

✓ Embeddings normalized


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Block 3 — Scoring functions (copy from rank_v2_latest.py)
# ─────────────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')  # add current dir to path

# Import all scoring functions directly from rank_v2_latest.py
from rank_v2_latest import (
    compute_feature_score,
    compute_semantic_score,
    compute_behavioral_score,
    compute_penalty,
    generate_reasoning,
    JD_RELEVANT_ASSESSMENTS,
    MUST_HAVE_KEYWORDS,
    NICE_TO_HAVE_KEYWORDS,
    SHIPPER_KEYWORDS,
    DEPTH_KEYWORDS,
    PRE_LLM_KEYWORDS
)

print('✓ Scoring functions imported from rank_v2_latest.py')
print(f'  Must-have groups  : {len(MUST_HAVE_KEYWORDS)}')
print(f'  Nice-to-have groups: {len(NICE_TO_HAVE_KEYWORDS)}')
print(f'  Shipper keywords  : {len(SHIPPER_KEYWORDS)}')
print(f'  Depth keywords    : {len(DEPTH_KEYWORDS)}')

✓ Scoring functions imported from rank_v2_latest.py
  Must-have groups  : 5
  Nice-to-have groups: 10
  Shipper keywords  : 76
  Depth keywords    : 43


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Block 4 — FAISS or NumPy cosine similarity + score all survivors
# ─────────────────────────────────────────────────────────────────────────────
import time

# Build candidate texts from v2 metadata (full_summary now available)
candidate_texts = []
for m in metadata:
    text = ' '.join([
        m.get('full_summary', m.get('summary_snippet', '')),  # v2: full summary
        m.get('current_title', ''),
        m.get('headline', ''),
        ' '.join(m.get('skill_names', [])),
        ' '.join(m.get('career_titles', []))
    ]).lower()
    candidate_texts.append(text)

print(f'✓ Candidate texts built')
print(f'  Sample length: {len(candidate_texts[surviving[0]])} chars')

# Cosine similarity — FAISS or NumPy
print('\nComputing cosine similarities...')
surviving_embeddings = embeddings_norm[surviving]

try:
    import faiss
    print('  Using FAISS IndexFlatIP')
    dim   = surviving_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(surviving_embeddings)
    jd_query = jd_embedding.reshape(1, -1)
    cosine_sims_raw, indices = index.search(jd_query, len(surviving))
    faiss_scores = np.zeros(len(surviving))
    for pos, orig_idx in enumerate(indices[0]):
        faiss_scores[orig_idx] = cosine_sims_raw[0][pos]
    cosine_sims = faiss_scores
except ImportError:
    print('  Using NumPy (identical results)')
    cosine_sims = surviving_embeddings @ jd_embedding

# Score all survivors
print(f'\nScoring {len(surviving):,} survivors...')
start  = time.time()
scored = []

for rank_idx, i in enumerate(surviving):
    m    = metadata[i]
    text = candidate_texts[i]

    feat_score, _                    = compute_feature_score(m)
    sem_score, must_hits, nice_hits  = compute_semantic_score(text, float(cosine_sims[rank_idx]))
    beh_score, beh_breakdown         = compute_behavioral_score(m)

    raw_score   = 0.40 * feat_score + 0.35 * sem_score + 0.25 * beh_score
    penalty, penalty_flags = compute_penalty(m)
    final_score = round(raw_score * penalty, 4)

    scored.append({
        'idx':          i,
        'm':            m,
        'final_score':  final_score,
        'feat_score':   round(feat_score, 4),
        'sem_score':    round(sem_score,  4),
        'beh_score':    round(beh_score,  4),
        'must_hits':    must_hits,
        'nice_hits':    nice_hits,
        'penalty':      round(penalty, 4),
        'penalty_flags': penalty_flags,
        'beh_breakdown': beh_breakdown,
    })

scored.sort(key=lambda x: (-x['final_score'], x['m']['candidate_id']))
print(f'✓ Scoring complete in {time.time()-start:.1f}s')

print(f'\nTop 10 preview:')
for rank, entry in enumerate(scored[:10], 1):
    m = entry['m']
    print(f'  [{rank:2d}] {m["candidate_id"]} — {entry["final_score"]:.4f} '
          f'| {m["current_title"][:28]:28s} | must:{entry["must_hits"]}/5 '
          f'| feat:{entry["feat_score"]:.3f} sem:{entry["sem_score"]:.3f} beh:{entry["beh_score"]:.3f} '
          f'| {m["location"].split(",")[0]:12s} | resp:{m["recruiter_response_rate"]:.0%}')

✓ Candidate texts built
  Sample length: 842 chars

Computing cosine similarities...
  Using NumPy (identical results)

Scoring 1,126 survivors...
✓ Scoring complete in 0.4s

Top 10 preview:
  [ 1] CAND_0018499 — 0.9480 | Senior Machine Learning Engi | must:5/5 | feat:0.956 sem:1.030 beh:0.821 | Noida        | resp:61%
  [ 2] CAND_0042506 — 0.8298 | Search Engineer              | must:4/5 | feat:0.929 sem:0.779 beh:0.742 | Mumbai       | resp:48%
  [ 3] CAND_0066690 — 0.8238 | ML Engineer                  | must:4/5 | feat:0.908 sem:0.730 beh:0.821 | Gurgaon      | resp:88%
  [ 4] CAND_0068811 — 0.8228 | Applied ML Engineer          | must:5/5 | feat:0.905 sem:0.843 beh:0.663 | Pune         | resp:42%
  [ 5] CAND_0007009 — 0.8185 | Recommendation Systems Engin | must:4/5 | feat:0.933 sem:0.821 beh:0.632 | Noida        | resp:62%
  [ 6] CAND_0009024 — 0.8156 | Search Engineer              | must:5/5 | feat:0.922 sem:0.860 beh:0.583 | Chennai      | resp:46%
  [ 7] CAND_0046459 — 0.7934 

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Block 5 — Inspect any candidate in detail
# Change candidate_id to inspect any ranked candidate
# ─────────────────────────────────────────────────────────────────────────────
def inspect(rank_position):
    entry = scored[rank_position - 1]
    m     = entry['m']
    print(f'=== Rank {rank_position}: {m["candidate_id"]} ===')
    print(f'Title        : {m["current_title"]}')
    print(f'Company      : {m["current_company"]}')
    print(f'Location     : {m["location"]} ({m["location_bucket"]})')
    print(f'Experience   : {m["years_of_experience"]} years')
    print(f'Notice       : {m["notice_period_days"]} days')
    print(f'Consulting % : {m["consulting_pct"]:.0%}')
    print(f'AI exp months: {m["ai_experience_months"]}')
    print(f'Avg tenure   : {m.get("avg_tenure_months", 0):.0f} months')
    print(f'FAANG only   : {m.get("is_faang_only", False)}')
    print(f'Company size : {m.get("company_size_score", 0):.1f}')
    print(f'GitHub       : {m["github_activity_score"]}')
    print(f'Response rate: {m["recruiter_response_rate"]:.0%}')
    print(f'Completeness : {m["profile_completeness_score"]:.0f}%')
    print(f'Summary qual : {m.get("summary_quality", 0):.2f}')
    print(f'Skills       : {m["skill_names"]}')
    print(f'Career titles: {m["career_titles"]}')
    print(f'Companies    : {m["career_companies"]}')
    print(f'Full summary : {m.get("full_summary", "")[:300]}...')
    print(f'\nScores:')
    print(f'  Feature : {entry["feat_score"]}')
    print(f'  Semantic: {entry["sem_score"]} (must:{entry["must_hits"]}/5 nice:{entry["nice_hits"]})')
    print(f'  Behav   : {entry["beh_score"]}')
    print(f'  Penalty : {entry["penalty"]} {entry["penalty_flags"]}')
    print(f'  FINAL   : {entry["final_score"]:.4f}')

# Inspect top 5
for r in range(1, 6):
    inspect(r)
    print()

=== Rank 1: CAND_0018499 ===
Title        : Senior Machine Learning Engineer
Company      : Zomato
Location     : Noida, Uttar Pradesh (preferred)
Experience   : 7.2 years
Notice       : 15 days
Consulting % : 0%
AI exp months: 86
Avg tenure   : 29 months
FAANG only   : False
Company size : 0.7
GitHub       : 94.8
Response rate: 61%
Completeness : 98%
Summary qual : 0.87
Skills       : ['Deep Learning', 'Weaviate', 'Recommendation Systems', 'scikit-learn', 'Diffusion Models', 'Pinecone', 'Information Retrieval', 'Milvus', 'QLoRA', 'RAG']
Career titles: ['Senior Machine Learning Engineer', 'Staff Machine Learning Engineer', 'Senior Machine Learning Engineer']
Companies    : ['Zomato', 'Google', 'Flipkart']
Full summary : Senior AI engineer with 7.2 years of hands-on experience building production ML systems, with a focus on search, retrieval, and ranking. Most recently, I designed the company's first hybrid retrieval system combining BM25 with dense vector recall, serving 50M+ queries p

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# Block 6 — Generate CSV and validate
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

# R14: Filter out 0/5 must-haves from bottom of top 100
strong   = [e for e in scored if e['must_hits'] >= 1]
weak     = [e for e in scored if e['must_hits'] == 0]
top_pool = strong[:100] if len(strong) >= 100 else strong + weak[:100 - len(strong)]
top      = top_pool[:100]

print(f'Candidates with 0/5 must-haves filtered: {len(weak)}')
print(f'Top 100 selected from {len(strong)} strong candidates')

rows = []
for rank, entry in enumerate(top, 1):
    m = entry['m']
    reasoning = generate_reasoning(
        m=m, rank=rank,
        must_hits=entry['must_hits'],
        nice_hits=entry['nice_hits'],
        beh_scores=entry['beh_breakdown'],
        penalty_flags=entry['penalty_flags'],
        final_score=entry['final_score']
    )
    rows.append({
        'candidate_id': m['candidate_id'],
        'rank':         rank,
        'score':        round(entry['final_score'], 4),
        'reasoning':    reasoning
    })

# Validate
print('\nValidating...')
assert len(rows) == 100
assert len(set(r['candidate_id'] for r in rows)) == 100
assert all(r['reasoning'] is not None for r in rows)
scores_list = [r['score'] for r in rows]
assert all(scores_list[i] >= scores_list[i+1] for i in range(len(scores_list)-1))
print('  ✓ 100 rows')
print('  ✓ No duplicates')
print('  ✓ No None reasoning')
print('  ✓ Scores monotonically decreasing')
print(f'  Score range: {scores_list[0]:.4f} -> {scores_list[-1]:.4f}')

# Save CSV — write in binary mode to guarantee CRLF on all platforms
out_path = Path(f'{OUTPUT_DIR}/submission_v2.csv')
with open(out_path, 'wb') as f:  # binary mode — no line ending conversion
    content = 'candidate_id,rank,score,reasoning\r\n'
    for r in rows:
        reasoning_escaped = r['reasoning'].replace('"', '""')
        content += f'{r["candidate_id"]},{r["rank"]},{r["score"]},"{reasoning_escaped}"\r\n'
    f.write(content.encode('utf-8'))

print(f'\n✅ Saved: {out_path}')
print(f'   File size: {out_path.stat().st_size/1e3:.1f} KB')

# Verify CRLF
with open(out_path, 'rb') as f:
    raw = f.read(200)
print(f'   CRLF confirmed: {b"\\r\\n" in raw}')

Candidates with 0/5 must-haves filtered: 316
Top 100 selected from 810 strong candidates

Validating...
  ✓ 100 rows
  ✓ No duplicates
  ✓ No None reasoning
  ✓ Scores monotonically decreasing
  Score range: 0.9480 -> 0.5373

✅ Saved: outputs_v2\submission_v2.csv
   File size: 19.4 KB
   CRLF confirmed: False


In [ ]:
### END ###